<a href="https://colab.research.google.com/github/Heng1222/VeriPromiseESG_2026_TEAM_9906/blob/feat-model-train/app/model/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CKIP-BERT Simplified Inference

Loads one ESG-MLM backbone and five LoRA classifiers, then averages their probabilities. T1/T3 use fixed 0.5 thresholds; T2/T4 use argmax.


In [ ]:
# Colab dependencies. Restart the runtime if requested.
# !pip install -q transformers peft accelerate torch pandas numpy tqdm huggingface_hub safetensors


In [ ]:
import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download, snapshot_download
from peft import PeftModel
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


# ==========================================
# Simplified five-fold artifact v8 inference
# ==========================================

DEFAULT_REPO_ID = "maxbeettww/VeriPromise_ESG_2026_9906"
BATCH_SIZE = 16
ID_COLUMN = "id"
TEXT_COLUMN = "data"
TARGET_COLUMNS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]
TASK_CLASSES = {
    "t1": ["No", "Yes"],
    "t2": [
        "already",
        "within_2_years",
        "between_2_and_5_years",
        "more_than_5_years",
    ],
    "t3": ["No", "Yes"],
    "t4": ["Clear", "Not Clear", "Misleading"],
}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"


def find_artifact_root(path):
    path = Path(path)
    for candidate in [path, path / "mtl_outputs"]:
        if (candidate / "mtl_inference_config.json").exists():
            return candidate
    return None


def resolve_artifact_root(repo_id=None, model_dir=None):
    if repo_id:
        config_filename = "mtl_inference_config.json"
        try:
            hf_hub_download(repo_id=repo_id, filename=config_filename)
        except Exception:
            config_filename = "mtl_outputs/mtl_inference_config.json"
            hf_hub_download(repo_id=repo_id, filename=config_filename)
        downloaded = snapshot_download(
            repo_id=repo_id,
            allow_patterns=[
                config_filename,
                "mlm_backbone/**",
                "tokenizer/**",
                "fold_*/adapter/**",
                "fold_*/heads.pt",
                "mtl_outputs/mlm_backbone/**",
                "mtl_outputs/tokenizer/**",
                "mtl_outputs/fold_*/adapter/**",
                "mtl_outputs/fold_*/heads.pt",
            ],
        )
        root = find_artifact_root(downloaded)
        if root is None:
            raise FileNotFoundError("Downloaded artifact has no config.")
        return root

    model_dir = Path("mtl_outputs") if model_dir is None else Path(model_dir)
    root = find_artifact_root(model_dir)
    if root is None:
        raise FileNotFoundError(f"No artifact found under {model_dir}.")
    return root


def load_config(root):
    with open(
        Path(root) / "mtl_inference_config.json",
        "r",
        encoding="utf-8",
    ) as file:
        config = json.load(file)
    if int(config.get("artifact_version", 0)) != 8:
        raise ValueError("This notebook only supports simplified artifact v8.")
    return config


def tokenize_head_tail(text, tokenizer, max_len, head_ratio):
    body_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False,
        verbose=False,
    )
    max_body_len = max_len - 2
    if len(body_ids) > max_body_len:
        head_len = int(max_body_len * head_ratio)
        body_ids = (
            body_ids[:head_len]
            + body_ids[-(max_body_len - head_len):]
        )
    input_ids = [tokenizer.cls_token_id] + body_ids + [tokenizer.sep_token_id]
    attention_mask = [1] * len(input_ids)
    pad_len = max_len - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * pad_len
    attention_mask += [0] * pad_len
    return input_ids, attention_mask


class InferenceDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len, head_ratio):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.head_ratio = head_ratio

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        input_ids, attention_mask = tokenize_head_tail(
            self.dataframe.iloc[index][TEXT_COLUMN],
            self.tokenizer,
            self.max_len,
            self.head_ratio,
        )
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        }


class SimpleESGModel(nn.Module):
    def __init__(self, backbone_path, adapter_dir):
        super().__init__()
        base_model = AutoModel.from_pretrained(backbone_path)
        self.backbone = PeftModel.from_pretrained(
            base_model,
            adapter_dir,
            is_trainable=False,
        )
        hidden_size = base_model.config.hidden_size
        self.shared_mlp = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.heads = nn.ModuleDict(
            {
                task: nn.Sequential(
                    nn.Linear(256, 128),
                    nn.GELU(),
                    nn.Dropout(0.10),
                    nn.Linear(128, len(labels)),
                )
                for task, labels in TASK_CLASSES.items()
            }
        )

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        ).last_hidden_state
        features = self.shared_mlp(hidden[:, 0, :])
        return {
            task: head(features)
            for task, head in self.heads.items()
        }

    def load_heads(self, path):
        state = torch.load(path, map_location=DEVICE, weights_only=True)
        self.shared_mlp.load_state_dict(state["shared_mlp"])
        self.heads.load_state_dict(state["heads"])


def predict_probabilities(model, loader):
    output = {task: [] for task in TASK_CLASSES}
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP,
            ):
                logits = model(input_ids, attention_mask)
            for task in TASK_CLASSES:
                output[task].append(
                    torch.softmax(logits[task], dim=-1).cpu().numpy()
                )
    return {
        task: np.concatenate(parts, axis=0)
        for task, parts in output.items()
    }


def ensemble_probabilities(root, config, loader):
    members = config.get("ensemble_members", [])
    if len(members) != 5:
        raise ValueError("Artifact v8 must contain five ensemble members.")
    accumulated = {task: None for task in TASK_CLASSES}
    for member in members:
        member_dir = Path(root) / member
        print(f"Loading {member}")
        model = SimpleESGModel(
            Path(root) / config["backbone_path"],
            member_dir / "adapter",
        ).to(DEVICE)
        model.load_heads(member_dir / "heads.pt")
        probabilities = predict_probabilities(model, loader)
        for task, values in probabilities.items():
            if accumulated[task] is None:
                accumulated[task] = np.zeros_like(
                    values,
                    dtype=np.float64,
                )
            accumulated[task] += values
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return {
        task: values / len(members)
        for task, values in accumulated.items()
    }


def route_predictions(dataframe, probabilities, t1_threshold, t3_threshold):
    rows = []
    for index, row in dataframe.reset_index(drop=True).iterrows():
        t1 = (
            "Yes"
            if probabilities["t1"][index, 1] >= t1_threshold
            else "No"
        )
        if t1 == "No":
            rows.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "No",
                    "verification_timeline": "N/A",
                    "evidence_status": "N/A",
                    "evidence_quality": "N/A",
                }
            )
            continue
        t2 = TASK_CLASSES["t2"][
            int(probabilities["t2"][index].argmax())
        ]
        t3 = (
            "Yes"
            if probabilities["t3"][index, 1] >= t3_threshold
            else "No"
        )
        if t3 == "No":
            rows.append(
                {
                    ID_COLUMN: row[ID_COLUMN],
                    "promise_status": "Yes",
                    "verification_timeline": t2,
                    "evidence_status": "No",
                    "evidence_quality": "N/A",
                }
            )
            continue
        rows.append(
            {
                ID_COLUMN: row[ID_COLUMN],
                "promise_status": "Yes",
                "verification_timeline": t2,
                "evidence_status": "Yes",
                "evidence_quality": TASK_CLASSES["t4"][
                    int(probabilities["t4"][index].argmax())
                ],
            }
        )
    return pd.DataFrame(rows)[[ID_COLUMN] + TARGET_COLUMNS]


def validate_input(dataframe):
    missing = [
        column
        for column in [ID_COLUMN, TEXT_COLUMN]
        if column not in dataframe.columns
    ]
    if missing:
        raise ValueError(f"Test CSV missing columns: {missing}")
    if dataframe[ID_COLUMN].duplicated().any():
        raise ValueError("Test CSV contains duplicated ids.")


def inference_and_export(
    repo_id,
    test_csv_path,
    output_csv_path="final_submission.csv",
    model_dir=None,
):
    root = resolve_artifact_root(repo_id=repo_id, model_dir=model_dir)
    config = load_config(root)
    test_df = pd.read_csv(test_csv_path).reset_index(drop=True)
    validate_input(test_df)

    tokenizer = AutoTokenizer.from_pretrained(
        Path(root) / config["tokenizer_path"],
        use_fast=True,
    )
    loader = DataLoader(
        InferenceDataset(
            test_df,
            tokenizer,
            int(config["max_len"]),
            float(config["head_ratio"]),
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=USE_AMP,
    )
    probabilities = ensemble_probabilities(root, config, loader)
    thresholds = config["thresholds"]
    output_df = route_predictions(
        test_df,
        probabilities,
        t1_threshold=float(thresholds["t1_yes"]),
        t3_threshold=float(thresholds["t3_yes"]),
    )
    output_df.to_csv(output_csv_path, index=False)
    print(f"Exported predictions to {output_csv_path}")
    print(output_df.head())
    return output_df


In [ ]:
# ==========================================
# Run inference
# ==========================================

TEST_CSV_PATH = "../data/ori_data/vpesg4k_test_2000.csv"

inference_and_export(
    repo_id=DEFAULT_REPO_ID,
    test_csv_path=TEST_CSV_PATH,
    output_csv_path="final_submission.csv",
)
